# 1. The data and its missing values

**Question.** What is in the NHANES tables, which adults end up in the analysis, how is the heart
disease label built, and what is missing (and does the missingness itself carry information)?

How to read this notebook: each section runs a query or a calculation and prints the result. The
last two cells, *what this means* and *what is weak*, are code cells so that every number in their
sentences is computed, not typed. Lines starting with `Check:` compare a number recomputed here
with the one the pipeline wrote to `docs/results/results.json`.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from nhanes import nb, plots
from nhanes.features import CATEGORICAL, FEATURES, LABELS, NUMERIC, TARGET

ctx = nb.context()
df, R = ctx.df, ctx.R
pct = nb.pct
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.width", 160)
print("mode:", "synthetic fixture (CI)" if ctx.fixture else "real NHANES data")
print(f"{len(df):,} adults, {int(df[TARGET].sum()):,} reporting heart disease")

mode: real NHANES data
22,512 adults, 1,637 reporting heart disease


## 1.1 Who is in the analysis (a SQL query on the model schema)

In [2]:
funnel = ctx.sql("SELECT * FROM model.cohort_funnel ORDER BY cycle")
display(funnel)
nb.check("labelled adults", funnel.adults_with_label.sum(), R["data"]["n_total"])
dropped = funnel.adults_20_plus.sum() - funnel.adults_with_label.sum()
print(
    f"Adults dropped because the label is unknown (no yes, and at least one refused / don't know / missing): {dropped:,}"
)

,cycle,all_participants,adults_20_plus,adults_with_mcq,adults_with_label
0,G,9756,5560,5560,5538
1,H,10175,5769,5769,5752
2,I,9971,5719,5719,5688
3,J,9254,5569,5569,5534


Check: labelled adults: notebook 22512 vs results.json 22512 (matches)
Adults dropped because the label is unknown (no yes, and at least one refused / don't know / missing): 105


## 1.2 How refused and don't-know answers look in the raw table

The CDC codes refused as 7 and don't know as 9. If these were left as numbers they would look like
"more than yes". The raw table below is read straight from PostgreSQL (`raw.mcq`), before any cleaning.


In [3]:
codes = ctx.sql("""
    SELECT cycle,
           COUNT(*) FILTER (WHERE mcq160c = 1)          AS yes,
           COUNT(*) FILTER (WHERE mcq160c = 2)          AS no,
           COUNT(*) FILTER (WHERE mcq160c IN (7, 9))    AS refused_or_dont_know,
           COUNT(*) FILTER (WHERE mcq160c IS NULL)      AS not_asked_or_missing
    FROM raw.mcq GROUP BY cycle ORDER BY cycle
""")
display(codes)
model_codes = ctx.sql(
    "SELECT COUNT(*) AS n FROM model.heart_risk WHERE heart_disease NOT IN (0, 1)"
)
print("Rows in the model table whose label is not 0 or 1:", int(model_codes.n[0]))

,cycle,yes,no,refused_or_dont_know,not_asked_or_missing
0,G,196,5342,21,3805
1,H,232,5519,18,4001
2,I,244,5449,26,3856
3,J,265,5288,16,3328


Rows in the model table whose label is not 0 or 1: 0


## 1.3 What the label is made of

Label = yes to coronary heart disease, angina or heart attack. Congestive heart failure and stroke
are different conditions and are **not** part of the label. They are also never used as features,
because they are consequences of (or entangled with) the diagnosis we are predicting.


In [4]:
comp = df[["chd", "angina", "heart_attack"]].fillna(False).astype(bool)
cases = comp[df[TARGET] == 1]
combo = cases.value_counts().rename("adults").reset_index()
combo["share_of_cases"] = combo.adults / combo.adults.sum()
display(combo)
other = (
    df[["chf", "stroke"]]
    .fillna(False)
    .astype(bool)
    .groupby(df[TARGET])
    .mean()
    .rename(index={0: "no heart disease", 1: "heart disease"})
)
display(other.rename(columns={"chf": "share with heart failure", "stroke": "share with stroke"}))

,chd,angina,heart_attack,adults,share_of_cases
0,False,False,True,409,0.250
1,True,False,False,358,0.219
2,True,False,True,311,0.190
3,False,True,False,221,0.135
4,True,True,True,164,0.100
5,True,True,False,104,0.064
6,False,True,True,70,0.043


,share with heart failure,share with stroke
heart_disease,,
no heart disease,0.014,0.029
heart disease,0.297,0.184


## 1.4 Splits by survey cycle (earlier cycles train, a later one validates, the latest tests)

In [5]:
splits = pd.DataFrame(R["data"]["splits"]).T[
    ["cycles", "survey_years", "n", "positives", "prevalence"]
]
display(splits)
nb.show(plots.plot_prevalence_by_cycle(df))

,cycles,survey_years,n,positives,prevalence
train,"[G, H]","2011-2012, 2013-2014",11290,752,0.067
valid,[I],2015-2016,5688,428,0.075
test,[J],2017-2018,5534,457,0.083


<Figure size 715x418 with 1 Axes>

## 1.5 Missing values

In [6]:
nb.show(plots.plot_missingness(df))
miss = df[FEATURES].isna()
share_complete = float((~miss.any(axis=1)).mean())
nb.check(
    "share of adults with every feature",
    share_complete,
    R["data"]["missingness"]["complete_case_share"],
)
patterns = (
    miss.apply(lambda r: ", ".join(c for c in FEATURES if r[c]) or "nothing missing", axis=1)
    .value_counts(normalize=True)
    .head(6)
    .rename("share of adults")
    .to_frame()
)
display(patterns)

<Figure size 825x550 with 1 Axes>

Check: share of adults with every feature: notebook 0.322139 vs results.json 0.322139 (matches)


,share of adults
fasting_glucose,0.344
nothing missing,0.322
"fasting_glucose, drinks_per_day, ever_heavy_drinker",0.081
"drinks_per_day, ever_heavy_drinker",0.076
"bmi, waist_cm, sbp_mean, dbp_mean, total_chol, hdl, hba1c, fasting_glucose, drinks_per_day, ever_heavy_drinker",0.043
"total_chol, hdl, hba1c, fasting_glucose",0.022


## 1.6 Is missingness informative?

If people with a feature missing have a different disease rate from people with it present, then how
we fill in missing values can matter. Fisher's exact test per feature (a small p-value only says the
rates differ; it does not say why).


In [7]:
from scipy.stats import fisher_exact

rows = []
for c in FEATURES:
    m = df[c].isna()
    if m.sum() < 20 or (~m).sum() < 20:
        continue
    a, b = int(df.loc[m, TARGET].sum()), int((~df.loc[m, TARGET].astype(bool)).sum())
    c1, d1 = int(df.loc[~m, TARGET].sum()), int((~df.loc[~m, TARGET].astype(bool)).sum())
    p = fisher_exact([[a, b], [c1, d1]])[1]
    rows.append(
        {
            "feature": c,
            "n_missing": int(m.sum()),
            "rate_when_missing": df.loc[m, TARGET].mean(),
            "rate_when_present": df.loc[~m, TARGET].mean(),
            "p_value": p,
        }
    )
miss_tab = pd.DataFrame(rows).sort_values("p_value")
display(miss_tab)
informative = miss_tab[miss_tab.p_value < 0.05].feature.tolist()

,feature,n_missing,rate_when_missing,rate_when_present,p_value
1,waist_cm,2303,0.107,0.069,0.000
0,bmi,1271,0.102,0.071,0.000
9,ever_heavy_drinker,5830,0.063,0.076,0.001
8,drinks_per_day,5829,0.064,0.076,0.002
7,fasting_glucose,12625,0.070,0.076,0.056
3,dbp_mean,1873,0.084,0.072,0.057
2,sbp_mean,1824,0.079,0.072,0.240
4,total_chol,2258,0.076,0.072,0.494
5,hdl,2258,0.076,0.072,0.494
6,hba1c,1976,0.074,0.073,0.751


## What this means

In [8]:
ms = R["data"]["missingness"]
top_miss = max(ms["overall"], key=ms["overall"].get)
lab_share = ms["overall"]["total_chol"]
nb.say(
    f"- {pct(ms['examined_share'])} of adults were examined, and only {pct(share_complete)} have **every** feature, "
    f"mainly because **{LABELS.get(top_miss, top_miss)}** is missing for {pct(ms['overall'][top_miss])} "
    f"(it exists only for the fasting morning subsample).\n"
    f"- Total cholesterol is missing for {pct(lab_share)} of adults. The most common pattern other than "
    f"'nothing missing' is: {patterns.index[patterns.index != 'nothing missing'][0]} "
    f"({pct(patterns.loc[patterns.index[patterns.index != 'nothing missing'][0], 'share of adults'])} of adults).\n"
    f"- Features whose missing group has a different disease rate (p < 5%): "
    f"{', '.join(informative) if informative else 'none'}.\n"
    f"- Unknown labels removed {dropped:,} adults ({pct(dropped / funnel.adults_20_plus.sum(), 2)} of adults)."
)

- 95.7% of adults were examined, and only 32.2% have **every** feature, mainly because **Fasting glucose (mg/dL)** is missing for 56.1% (it exists only for the fasting morning subsample).
- Total cholesterol is missing for 10.0% of adults. The most common pattern other than 'nothing missing' is: fasting_glucose (34.4% of adults).
- Features whose missing group has a different disease rate (p < 5%): waist_cm, bmi, ever_heavy_drinker, drinks_per_day.
- Unknown labels removed 105 adults (0.46% of adults).

## What is weak

In [9]:
cc = R["missing_value_strategies"]["complete_case"]
nb.say(
    f"- Dropping people with any missing feature would keep only {pct(cc['n_train_complete'] / cc['n_train_total'])} of the "
    f"training rows, so the pipeline fills values in instead (strategies compared in notebook 3 and RESULTS.md section 7).\n"
    "- The alcohol questions skip people who did not drink in the past year; SQL sets their drinks per day to 0, "
    "which is an assumption about the skip pattern, not a measured value.\n"
    "- The label is self-reported: it depends on people remembering and reporting what a doctor said, and on having "
    "seen a doctor at all. People without healthcare access may be labelled healthy by mistake (the data cannot tell us how often).\n"
    "- A test of many features against the outcome will flag some by chance; treat the table above as a hint, not proof."
)

- Dropping people with any missing feature would keep only 32.1% of the training rows, so the pipeline fills values in instead (strategies compared in notebook 3 and RESULTS.md section 7).
- The alcohol questions skip people who did not drink in the past year; SQL sets their drinks per day to 0, which is an assumption about the skip pattern, not a measured value.
- The label is self-reported: it depends on people remembering and reporting what a doctor said, and on having seen a doctor at all. People without healthcare access may be labelled healthy by mistake (the data cannot tell us how often).
- A test of many features against the outcome will flag some by chance; treat the table above as a hint, not proof.